In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)
rng = np.random.default_rng(seed=666)

In [ ]:
x = rng.random((1024, 1024))
y = rng.random((1024, 1024))

In [ ]:
x_jax = jnp.array(x, dtype=jnp.float32)
y_jax = jnp.array(y, dtype=jnp.float32)
o_tpu_1 = jax.lax.dot(x_jax, y_jax, precision=jax.lax.Precision.DEFAULT)
o_tpu_2 = jax.lax.dot(x_jax, y_jax, precision=jax.lax.Precision.HIGH)
o_tpu_3 = jax.lax.dot(x_jax, y_jax, precision=jax.lax.Precision.HIGHEST)

In [ ]:
def calculate_ulp_bits(test_val, ref_val):
    """
    通过比特位差值计算 ULP 误差 (适用于 float32)
    """
    # 强制转为 f32 确保比特位对齐
    t_f32 = jnp.array(test_val, dtype=jnp.float32)
    r_f32 = jnp.array(ref_val, dtype=jnp.float32)

    # 将比特流解释为 int32
    # 注意：此方法假设两数同号且不含 NaN/Inf
    t_bits = jax.lax.bitcast_convert_type(t_f32, jnp.int32)
    r_bits = jax.lax.bitcast_convert_type(r_f32, jnp.int32)

    # 比特位的绝对差值即为 ULP 数量
    ulp_diff = jnp.abs(t_bits - r_bits)
    return ulp_diff


def compare_metrics(test_val, ref_val, name="Test"):
    # 转换为 float32 进行分析（因为你的实验核心是 f32 的硬件表现）
    # 但计算过程使用 float64 保证减法精确
    t_f32 = test_val.astype(jnp.float32)
    r_f32 = ref_val.astype(jnp.float32)

    abs_diff = jnp.abs(t_f32.astype(jnp.float64) - r_f32.astype(jnp.float64))

    # 计算 ULP 误差
    # eps 是当前量级下的最小单位
    # 公式：ULP 误差 = |test - ref| / eps(ref)
    # jnp.spacing(x) 返回的是 x 对应位置的 1 ULP 的实际大小
    ulp_diff = calculate_ulp_bits(t_f32, r_f32)
    max_ulp = jnp.max(ulp_diff)

    # 计算最大相对误差 (带 epsilon 保护)
    max_rel = jnp.max(abs_diff / (jnp.abs(r_f32.astype(jnp.float64)) + 1e-10))

    print(f"--- {name} vs Reference ---")
    print(f"Max Absolute Error: {jnp.max(abs_diff):.6e}")
    print(f"Max Relative Error: {max_rel:.6e}")
    print(f"Max ULP Error:      {max_ulp:.2f}")  # 通常显示为浮点个数
    print(f"RMSE:               {jnp.sqrt(jnp.mean(jnp.square(abs_diff))):.6e}")

In [ ]:
# 使用示例
compare_metrics(o_tpu_1, o_tpu_2, name="DEFAULT (o1 vs o2)")
compare_metrics(o_tpu_2, o_tpu_3, name="HIGH (o2 vs o3)")
compare_metrics(o_tpu_1, o_tpu_3, name="HIGHEST (o1 vs o3)")

--- DEFAULT (o1 vs o2) vs Reference ---
Max Absolute Error: 1.024780e-01
Max Relative Error: 4.056818e-04
Max ULP Error:      6615.00
RMSE:               2.210332e-02
--- HIGH (o2 vs o3) vs Reference ---
Max Absolute Error: 2.227783e-03
Max Relative Error: 8.509270e-06
Max ULP Error:      141.00
RMSE:               1.740027e-03
--- HIGHEST (o1 vs o3) vs Reference ---
Max Absolute Error: 1.042175e-01
Max Relative Error: 3.986877e-04
Max ULP Error:      6501.00
RMSE:               2.197602e-02


In [ ]:
gpu_data = np.load("jax_gpu.npz")
o_gpu_1 = gpu_data["default"]
o_gpu_2 = gpu_data["high"]
o_gpu_3 = gpu_data["highest"]

In [ ]:
compare_metrics(o_tpu_1, o_gpu_1, name="TPU (o_tpu_1_default) vs GPU (o_gpu_1_default)")
compare_metrics(o_tpu_2, o_gpu_2, name="TPU (o_tpu_2_high) vs GPU (o_gpu_2_high)")
compare_metrics(o_tpu_3, o_gpu_3, name="TPU (o_tpu_3_highest) vs GPU (o_gpu_3_highest)")

--- TPU (o_tpu_1_default) vs GPU (o_gpu_1_default) vs Reference ---
Max Absolute Error: 1.063080e-01
Max Relative Error: 4.272784e-04
Max ULP Error:      6967.00
RMSE:               2.253146e-02
--- TPU (o_tpu_2_high) vs GPU (o_gpu_2_high) vs Reference ---
Max Absolute Error: 1.388550e-02
Max Relative Error: 5.477142e-05
Max ULP Error:      910.00
RMSE:               3.066222e-03
--- TPU (o_tpu_3_highest) vs GPU (o_gpu_3_highest) vs Reference ---
Max Absolute Error: 1.525879e-04
Max Relative Error: 5.950060e-07
Max ULP Error:      9.00
RMSE:               3.251235e-05
